In [1]:
# # CELL 1: Install dependencies
# import subprocess, sys
# subprocess.check_call([
#     sys.executable, "-m", "pip", "install", "-q",
#     "huggingface_hub==0.23.4",
#     "diffusers==0.28.2",
#     "transformers==4.40.2",
#     "accelerate==0.30.1",
#     "peft==0.11.1",
#     "safetensors==0.4.3",
#     "flask==3.0.3",
#     "pyngrok==7.1.6",
# ])
# print('Installed successfully')
# import IPython
# IPython.Application.instance().kernel.do_shutdown(restart=True)

In [2]:
# CELL 2: GPU check
import torch
assert torch.cuda.is_available(), 'No GPU detected. Set Runtime > Change runtime type > T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'VRAM: {vram:.1f} GB')

GPU: Tesla T4
VRAM: 14.6 GB


In [3]:
# CELL 3: Load model
import torch, gc, json, tempfile, os
from diffusers import StableDiffusionXLPipeline, AutoencoderKL, DPMSolverMultistepScheduler
from peft import PeftModel
from huggingface_hub import hf_hub_download

SDXL_ID = 'stabilityai/stable-diffusion-xl-base-1.0'
VAE_ID  = 'madebyollin/sdxl-vae-fp16-fix'
LORA_ID = 'YoussefSouissi/celeba-lora'
DEVICE  = 'cuda'
DTYPE   = torch.float16

gc.collect()
torch.cuda.empty_cache()

print('[1/3] Loading VAE...')
vae = AutoencoderKL.from_pretrained(VAE_ID, torch_dtype=DTYPE)

print('[2/3] Loading SDXL base model...')
pipe = StableDiffusionXLPipeline.from_pretrained(
    SDXL_ID, vae=vae, torch_dtype=DTYPE, variant='fp16',
    use_safetensors=True, low_cpu_mem_usage=True
)

print('[3/3] Loading LoRA weights...')
config_path  = hf_hub_download(repo_id=LORA_ID, filename='adapter_config.json')
weights_path = hf_hub_download(repo_id=LORA_ID, filename='adapter_model.safetensors')

with open(config_path) as f:
    cfg = json.load(f)
for field in ['corda_config','eva_config','qalora_group_size','lora_bias',
              'target_parameters','trainable_token_indices','use_qalora','exclude_modules']:
    cfg.pop(field, None)

tmp = tempfile.mkdtemp()
with open(os.path.join(tmp, 'adapter_config.json'), 'w') as f:
    json.dump(cfg, f)
os.symlink(weights_path, os.path.join(tmp, 'adapter_model.safetensors'))

pipe.unet = PeftModel.from_pretrained(pipe.unet, tmp, is_trainable=False)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, use_karras_sigmas=True
)
pipe.to(DEVICE)
pipe.enable_attention_slicing(1)
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()

gc.collect()
torch.cuda.empty_cache()
print('Model ready.')

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/models/transformers/transformer_2d.py:34: FutureWarning: `Transformer2DModelOutput` is deprecated and will be removed in version 1.0.0. Importing `Transformer2DModelOutput` from `diffusers.models.transformer_2d` is deprecated and this will be removed in a future version. Please use `from diffusers.models.modeling_outputs import Transformer2DModelOutput`, instead.
  deprecate("Transformer2DModelOutput", "1.0.0", deprecation_message)


[1/3] Loading VAE...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

[3/3] Loading LoRA weights...


adapter_config.json:   0%|          | 0.00/736 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/23.3M [00:00<?, ?B/s]

Model ready.


In [4]:
# CELL 4: Flask server + ngrok tunnel
from flask import Flask, render_template_string, request, send_file
from pyngrok import ngrok, conf
from threading import Thread
from io import BytesIO
import os, hashlib, time, gc, torch

NEGATIVE = (
    'deformed face, mutation, extra nose, double face, two faces, '
    'blurry, low quality, ugly, disfigured, bad anatomy, '
    'extra limbs, missing nose, cropped face, out of frame, '
    'watermark, text, jpeg artifacts, oversaturated'
)

def make_seed(p):
    raw = f'{p}{time.time_ns()}{os.urandom(8).hex()}'
    return int(hashlib.sha256(raw.encode()).hexdigest(), 16) % (2**31)

def generate_image(prompt):
    seed = make_seed(prompt)
    with torch.no_grad():
        result = pipe(
            prompt.strip(),
            negative_prompt=NEGATIVE,
            generator=torch.Generator(device=DEVICE).manual_seed(seed),
            num_inference_steps=50,
            guidance_scale=9.0,
            height=1024,
            width=1024,
        )
    gc.collect()
    torch.cuda.empty_cache()
    return result.images[0], seed

HTML = '<!DOCTYPE html>\n<html lang="en">\n<head>\n<meta charset="UTF-8">\n<meta name="viewport" content="width=device-width, initial-scale=1.0">\n<title>Portrait Studio</title>\n<style>\n  *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }\n\n  :root {\n    --bg: #f6f5f0;\n    --surface: #ffffff;\n    --surface2: #f0eeea;\n    --border: #e2e0db;\n    --border-hover: #c8c5be;\n    --text: #1a1a18;\n    --text-muted: #6b6860;\n    --text-hint: #9c9a94;\n    --primary: #5147d4;\n    --primary-light: #ede9ff;\n    --primary-hover: #4038c0;\n    --accent: #c9a96e;\n    --destructive: #c0392b;\n    --radius: 10px;\n    --radius-sm: 6px;\n    --radius-full: 999px;\n    --shadow: 0 1px 3px rgba(0,0,0,0.06), 0 1px 2px rgba(0,0,0,0.04);\n    --shadow-md: 0 4px 12px rgba(0,0,0,0.08);\n  }\n\n  .dark {\n    --bg: #0f0f0e;\n    --surface: #1a1a18;\n    --surface2: #242422;\n    --border: #2e2e2b;\n    --border-hover: #444440;\n    --text: #f0eeea;\n    --text-muted: #9c9a94;\n    --text-hint: #6b6860;\n    --primary: #7b72f0;\n    --primary-light: #1e1a3a;\n    --primary-hover: #9088f4;\n    --accent: #c9a96e;\n    --destructive: #e05555;\n  }\n\n  body {\n    font-family: -apple-system, BlinkMacSystemFont, \'Segoe UI\', sans-serif;\n    background: var(--bg);\n    color: var(--text);\n    min-height: 100vh;\n    display: flex;\n    flex-direction: column;\n    font-size: 14px;\n    line-height: 1.5;\n    transition: background 0.2s, color 0.2s;\n  }\n\n  /* ── Header ── */\n  header {\n    position: sticky; top: 0; z-index: 10;\n    background: var(--bg);\n    border-bottom: 1px solid var(--border);\n    backdrop-filter: blur(8px);\n    -webkit-backdrop-filter: blur(8px);\n  }\n  .header-inner {\n    max-width: 1200px; margin: 0 auto;\n    padding: 10px 24px;\n    display: flex; align-items: center; justify-content: space-between;\n  }\n  .logo { display: flex; align-items: center; gap: 10px; }\n  .logo-icon {\n    width: 32px; height: 32px; border-radius: var(--radius-sm);\n    background: var(--primary-light);\n    display: flex; align-items: center; justify-content: center;\n    font-size: 16px;\n  }\n  .logo-text h1 { font-size: 14px; font-weight: 600; line-height: 1.2; }\n  .logo-text p { font-size: 11px; color: var(--text-muted); }\n  .header-right { display: flex; align-items: center; gap: 10px; }\n  .gpu-badge {\n    display: flex; align-items: center; gap: 6px;\n    font-size: 11px; color: var(--text-muted);\n    background: var(--surface2);\n    border: 1px solid var(--border);\n    border-radius: var(--radius-full);\n    padding: 4px 10px;\n  }\n  .gpu-dot { width: 6px; height: 6px; border-radius: 50%; background: #22c55e; }\n  .theme-btn {\n    width: 34px; height: 34px; border-radius: var(--radius-sm);\n    border: 1px solid var(--border);\n    background: var(--surface);\n    color: var(--text-muted);\n    cursor: pointer; font-size: 15px;\n    display: flex; align-items: center; justify-content: center;\n    transition: all 0.15s;\n  }\n  .theme-btn:hover { border-color: var(--border-hover); color: var(--text); }\n\n  /* ── Main layout ── */\n  main {\n    max-width: 1200px; margin: 0 auto; width: 100%;\n    padding: 24px;\n    display: grid;\n    grid-template-columns: 420px 1fr;\n    gap: 24px;\n    flex: 1;\n  }\n  @media (max-width: 768px) {\n    main { grid-template-columns: 1fr; padding: 16px; }\n  }\n\n  /* ── Left panel ── */\n  .left-panel { display: flex; flex-direction: column; gap: 16px; }\n\n  /* ── Mode switcher ── */\n  .mode-switch {\n    display: flex; gap: 4px;\n    background: var(--surface2);\n    border: 1px solid var(--border);\n    border-radius: var(--radius);\n    padding: 4px;\n  }\n  .mode-btn {\n    flex: 1; padding: 7px 12px;\n    border: none; border-radius: var(--radius-sm);\n    background: transparent;\n    color: var(--text-muted);\n    font-size: 12px; font-weight: 500;\n    cursor: pointer; transition: all 0.15s;\n  }\n  .mode-btn.active {\n    background: var(--surface);\n    color: var(--text);\n    box-shadow: var(--shadow);\n  }\n\n  /* ── Panel sections ── */\n  .panel {\n    background: var(--surface);\n    border: 1px solid var(--border);\n    border-radius: var(--radius);\n    padding: 16px;\n    display: flex; flex-direction: column; gap: 12px;\n  }\n  .section-label {\n    font-size: 10px; font-weight: 600;\n    text-transform: uppercase; letter-spacing: 0.1em;\n    color: var(--text-muted);\n  }\n\n  /* ── Textarea ── */\n  textarea {\n    width: 100%; min-height: 120px;\n    padding: 10px 12px;\n    background: var(--bg);\n    border: 1px solid var(--border);\n    border-radius: var(--radius-sm);\n    color: var(--text);\n    font-size: 12px; font-family: \'SF Mono\', \'Fira Code\', monospace;\n    line-height: 1.6; resize: vertical;\n    transition: border-color 0.15s;\n  }\n  textarea:focus { outline: none; border-color: var(--primary); }\n  textarea::placeholder { color: var(--text-hint); }\n  .prompt-hint { font-size: 11px; color: var(--text-muted); }\n  .prompt-hint span { color: var(--text); }\n\n  /* ── Hints toggle ── */\n  .hints-toggle {\n    width: 100%; display: flex; align-items: center; justify-content: space-between;\n    padding: 8px 12px;\n    background: transparent;\n    border: 1px solid var(--border);\n    border-radius: var(--radius-sm);\n    color: var(--text-muted);\n    font-size: 12px; font-weight: 500;\n    cursor: pointer; transition: all 0.15s;\n  }\n  .hints-toggle:hover { border-color: var(--border-hover); color: var(--text); background: var(--surface2); }\n  .hints-toggle .arrow { transition: transform 0.2s; font-size: 10px; }\n  .hints-toggle.open .arrow { transform: rotate(180deg); }\n\n  .hints-panel {\n    background: var(--surface2);\n    border: 1px solid var(--border);\n    border-radius: var(--radius-sm);\n    padding: 12px;\n    display: none; flex-direction: column; gap: 10px;\n  }\n  .hints-panel.open { display: flex; }\n  .hint-category-label {\n    font-size: 10px; font-weight: 600;\n    text-transform: uppercase; letter-spacing: 0.1em;\n    color: var(--text-muted); margin-bottom: 4px;\n  }\n  .hint-tags { display: flex; flex-wrap: wrap; gap: 6px; }\n  .hint-tag {\n    padding: 3px 10px;\n    background: transparent;\n    border: 1px solid var(--border);\n    border-radius: var(--radius-full);\n    color: var(--text-muted);\n    font-size: 11px; cursor: pointer; transition: all 0.15s;\n  }\n  .hint-tag:hover { border-color: var(--primary); background: var(--primary-light); color: var(--primary); }\n\n  /* ── Examples ── */\n  .example-list { display: flex; flex-direction: column; gap: 6px; }\n  .example-btn {\n    width: 100%; display: flex; align-items: flex-start; gap: 8px;\n    padding: 8px 10px;\n    background: transparent;\n    border: 1px solid var(--border);\n    border-radius: var(--radius-sm);\n    color: var(--text-muted);\n    font-size: 11px; text-align: left; cursor: pointer; transition: all 0.15s;\n    line-height: 1.5;\n  }\n  .example-btn:hover { border-color: var(--primary); background: var(--primary-light); color: var(--text); }\n  .example-icon { flex-shrink: 0; margin-top: 1px; opacity: 0.5; font-size: 12px; }\n  .example-text { display: -webkit-box; -webkit-line-clamp: 2; -webkit-box-orient: vertical; overflow: hidden; }\n\n  /* ── Separator ── */\n  .sep { height: 1px; background: var(--border); }\n\n  /* ── Builder selects ── */\n  .builder-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 10px; }\n  .select-group { display: flex; flex-direction: column; gap: 4px; }\n  .select-wrapper { position: relative; }\n  select {\n    width: 100%; padding: 7px 28px 7px 10px;\n    background: var(--bg);\n    border: 1px solid var(--border);\n    border-radius: var(--radius-sm);\n    color: var(--text);\n    font-size: 12px; cursor: pointer; appearance: none;\n    transition: border-color 0.15s;\n  }\n  select:focus { outline: none; border-color: var(--primary); }\n  .select-arrow {\n    position: absolute; right: 8px; top: 50%; transform: translateY(-50%);\n    pointer-events: none; color: var(--text-muted); font-size: 10px;\n  }\n\n  /* ── Prompt preview ── */\n  .prompt-preview {\n    background: var(--surface2);\n    border: 1px solid var(--border);\n    border-radius: var(--radius-sm);\n    padding: 10px 12px;\n    font-size: 11px; font-family: \'SF Mono\', \'Fira Code\', monospace;\n    color: var(--text-muted); line-height: 1.6;\n    word-break: break-word;\n  }\n\n  /* ── Generate button ── */\n  .generate-btn {\n    width: 100%; padding: 12px;\n    background: var(--primary);\n    color: #fff;\n    border: none; border-radius: var(--radius);\n    font-size: 13px; font-weight: 600;\n    cursor: pointer; transition: all 0.2s;\n    display: flex; align-items: center; justify-content: center; gap: 8px;\n  }\n  .generate-btn:hover:not(:disabled) { background: var(--primary-hover); transform: translateY(-1px); box-shadow: var(--shadow-md); }\n  .generate-btn:disabled { opacity: 0.5; cursor: not-allowed; transform: none; box-shadow: none; }\n  .spinner-sm {\n    width: 16px; height: 16px; border-radius: 50%;\n    border: 2px solid rgba(255,255,255,0.3);\n    border-top-color: #fff;\n    animation: spin 0.7s linear infinite;\n  }\n\n  /* ── Hardware info ── */\n  .hw-grid {\n    display: grid; grid-template-columns: 1fr 1fr;\n    gap: 4px 12px; font-size: 11px;\n  }\n  .hw-key { color: var(--text-muted); }\n  .hw-val { text-align: right; font-weight: 500; }\n  .profile-high { color: #22c55e; }\n  .profile-mid { color: #3b82f6; }\n  .profile-low { color: #f59e0b; }\n  .profile-cpu { color: #ef4444; }\n\n  /* ── Right panel ── */\n  .right-panel { display: flex; flex-direction: column; gap: 16px; }\n  .canvas-wrap {\n    background: var(--surface);\n    border: 1px solid var(--border);\n    border-radius: var(--radius);\n    aspect-ratio: 1;\n    display: flex; align-items: center; justify-content: center;\n    overflow: hidden; position: relative;\n  }\n  .canvas-empty {\n    display: flex; flex-direction: column; align-items: center; gap: 12px;\n    color: var(--text-hint); text-align: center;\n  }\n  .canvas-empty-icon { font-size: 48px; opacity: 0.2; }\n  .canvas-empty-text { font-size: 11px; text-transform: uppercase; letter-spacing: 0.1em; }\n  .canvas-generating {\n    position: absolute; inset: 0;\n    background: rgba(var(--bg-rgb, 246,245,240), 0.85);\n    backdrop-filter: blur(4px);\n    display: none; flex-direction: column;\n    align-items: center; justify-content: center; gap: 12px; z-index: 5;\n  }\n  .dark .canvas-generating { background: rgba(15,15,14,0.85); }\n  .canvas-generating.visible { display: flex; }\n  .spinner-lg {\n    width: 32px; height: 32px; border-radius: 50%;\n    border: 2px solid var(--border);\n    border-top-color: var(--primary);\n    animation: spin 0.8s linear infinite;\n  }\n  .canvas-generating p { font-size: 11px; text-transform: uppercase; letter-spacing: 0.1em; color: var(--text-muted); }\n  .canvas-generating small { font-size: 11px; color: var(--text-hint); }\n  #generated-img { width: 100%; height: 100%; object-fit: cover; display: none; }\n  .error-box {\n    display: none; flex-direction: column; align-items: center; gap: 8px;\n    padding: 24px; text-align: center;\n  }\n  .error-box.visible { display: flex; }\n  .error-icon { font-size: 28px; }\n  .error-title { font-size: 13px; font-weight: 600; color: var(--destructive); }\n  .error-msg { font-size: 11px; color: var(--text-muted); }\n\n  /* ── Image meta ── */\n  .image-meta {\n    display: none; align-items: center; justify-content: space-between;\n    background: var(--surface);\n    border: 1px solid var(--border);\n    border-radius: var(--radius);\n    padding: 10px 14px;\n  }\n  .image-meta.visible { display: flex; }\n  .meta-info { display: flex; gap: 16px; font-size: 11px; color: var(--text-muted); }\n  .meta-info strong { color: var(--text); font-weight: 500; }\n  .download-btn {\n    display: flex; align-items: center; gap: 6px;\n    padding: 6px 12px;\n    background: transparent;\n    border: 1px solid var(--border);\n    border-radius: var(--radius-sm);\n    color: var(--text-muted);\n    font-size: 11px; font-weight: 500;\n    cursor: pointer; transition: all 0.15s; text-decoration: none;\n  }\n  .download-btn:hover { border-color: var(--border-hover); color: var(--text); }\n\n  @keyframes spin { to { transform: rotate(360deg); } }\n</style>\n</head>\n<body>\n\n<!-- Header -->\n<header>\n  <div class="header-inner">\n    <div class="logo">\n      <div class="logo-icon">✦</div>\n      <div class="logo-text">\n        <h1>Portrait Studio</h1>\n        <p>SDXL · LoRA CelebA · Online</p>\n      </div>\n    </div>\n    <div class="header-right">\n      <div class="gpu-badge">\n        <div class="gpu-dot"></div>\n        <span>Colab GPU</span>\n      </div>\n      <button class="theme-btn" onclick="toggleTheme()" id="themeBtn" title="Toggle theme">🌙</button>\n    </div>\n  </div>\n</header>\n\n<!-- Main -->\n<main>\n  <!-- Left panel -->\n  <div class="left-panel">\n\n    <!-- Mode switcher -->\n    <div class="mode-switch">\n      <button class="mode-btn active" id="btn-manual" onclick="setMode(\'manual\')">Manual input</button>\n      <button class="mode-btn" id="btn-builder" onclick="setMode(\'builder\')">Prompt builder</button>\n    </div>\n\n    <!-- Manual mode -->\n    <div id="panel-manual" class="panel">\n      <div>\n        <div class="section-label" style="margin-bottom:8px">Describe your portrait</div>\n        <textarea id="prompt-input" placeholder="a hyperrealistic portrait photo of a young woman with brown hair, soft studio lighting, detailed skin texture, 8k resolution, sharp focus…"></textarea>\n        <p class="prompt-hint">Structure: <span>[shot type] of [subject] with [hair], [expression], [lighting], [quality]</span></p>\n      </div>\n\n      <!-- Hints -->\n      <button class="hints-toggle" id="hints-toggle" onclick="toggleHints()">\n        <span>Keyword hints — click to add</span>\n        <span class="arrow">▼</span>\n      </button>\n      <div class="hints-panel" id="hints-panel">\n        <div id="hints-content"></div>\n      </div>\n\n      <div class="sep"></div>\n\n      <!-- Examples -->\n      <div>\n        <div class="section-label" style="margin-bottom:8px">Example prompts</div>\n        <div class="example-list" id="examples-list"></div>\n      </div>\n    </div>\n\n    <!-- Builder mode -->\n    <div id="panel-builder" class="panel" style="display:none">\n      <div class="builder-grid" id="builder-grid"></div>\n      <div class="sep"></div>\n      <div>\n        <div class="section-label" style="margin-bottom:8px">Generated prompt preview</div>\n        <div class="prompt-preview" id="builder-preview">Select options above to build your prompt.</div>\n      </div>\n    </div>\n\n    <div class="sep"></div>\n\n    <!-- Generate button -->\n    <button class="generate-btn" id="generate-btn" onclick="handleGenerate()">\n      <span>✦</span>\n      <span id="btn-label">Generate Portrait</span>\n    </button>\n\n    <!-- Hardware info -->\n    <div class="panel">\n      <div class="section-label">System info</div>\n      <div class="hw-grid">\n        <span class="hw-key">Mode</span>\n        <span class="hw-val">Google Colab</span>\n        <span class="hw-key">GPU</span>\n        <span class="hw-val">T4 (16 GB)</span>\n        <span class="hw-key">Resolution</span>\n        <span class="hw-val">1024×1024</span>\n        <span class="hw-key">Steps</span>\n        <span class="hw-val">50 · DPMSolver</span>\n      </div>\n    </div>\n\n  </div>\n\n  <!-- Right panel -->\n  <div class="right-panel">\n    <div class="canvas-wrap" id="canvas-wrap">\n      <!-- Empty state -->\n      <div class="canvas-empty" id="canvas-empty">\n        <div class="canvas-empty-icon">◻</div>\n        <div class="canvas-empty-text">Your portrait will appear here</div>\n      </div>\n\n      <!-- Generating overlay -->\n      <div class="canvas-generating" id="canvas-generating">\n        <div class="spinner-lg"></div>\n        <p>Generating portrait…</p>\n        <small>30 – 60 seconds</small>\n      </div>\n\n      <!-- Error state -->\n      <div class="error-box" id="error-box">\n        <div class="error-icon">⚠</div>\n        <div class="error-title">Generation failed</div>\n        <div class="error-msg" id="error-msg"></div>\n      </div>\n\n      <!-- Generated image -->\n      <img id="generated-img" alt="Generated portrait" />\n    </div>\n\n    <!-- Image metadata + download -->\n    <div class="image-meta" id="image-meta">\n      <div class="meta-info">\n        <span><strong id="meta-res">1024×1024</strong> px</span>\n        <span>Seed <strong id="meta-seed">—</strong></span>\n        <span><strong id="meta-time">—</strong> s</span>\n      </div>\n      <a class="download-btn" id="download-btn" href="#" download="portrait.png">\n        ↓ Download\n      </a>\n    </div>\n  </div>\n</main>\n\n<script>\n// ── Theme ───────────────────────────────────────────────────────────────────\nlet isDark = window.matchMedia(\'(prefers-color-scheme: dark)\').matches;\nfunction applyTheme() {\n  document.body.classList.toggle(\'dark\', isDark);\n  document.getElementById(\'themeBtn\').textContent = isDark ? \'☀\' : \'🌙\';\n}\nfunction toggleTheme() { isDark = !isDark; applyTheme(); }\napplyTheme();\n\n// ── Mode ────────────────────────────────────────────────────────────────────\nlet currentMode = \'manual\';\nfunction setMode(m) {\n  currentMode = m;\n  document.getElementById(\'panel-manual\').style.display  = m === \'manual\'  ? \'flex\' : \'none\';\n  document.getElementById(\'panel-builder\').style.display = m === \'builder\' ? \'flex\' : \'none\';\n  document.getElementById(\'btn-manual\').classList.toggle(\'active\', m === \'manual\');\n  document.getElementById(\'btn-builder\').classList.toggle(\'active\', m === \'builder\');\n}\n\n// ── Hints ───────────────────────────────────────────────────────────────────\nconst HINTS = [\n  { label: \'Shot type\',  tags: [\'portrait photo\',\'cinematic close-up\',\'studio portrait\',\'close-up portrait\',\'editorial portrait\'] },\n  { label: \'Subject\',    tags: [\'a young woman\',\'a young man\',\'a middle-aged woman\',\'an elderly man\',\'a teenage girl\'] },\n  { label: \'Hair\',       tags: [\'brown hair\',\'blonde hair\',\'black hair\',\'curly hair\',\'wavy hair\',\'short hair\'] },\n  { label: \'Expression\', tags: [\'warm smile\',\'serious expression\',\'laughing\',\'neutral expression\',\'kind eyes\'] },\n  { label: \'Lighting\',   tags: [\'soft studio lighting\',\'dramatic side lighting\',\'natural daylight\',\'golden hour\',\'warm indoor lighting\'] },\n  { label: \'Quality\',    tags: [\'8k resolution\',\'4k resolution\',\'sharp focus\',\'highly detailed\',\'shallow depth of field\',\'photorealistic\'] },\n];\n\nfunction buildHints() {\n  const container = document.getElementById(\'hints-content\');\n  container.innerHTML = HINTS.map(cat => `\n    <div style="margin-bottom:10px">\n      <div class="hint-category-label">${cat.label}</div>\n      <div class="hint-tags">\n        ${cat.tags.map(t => `<button class="hint-tag" onclick="appendTag(\'${t}\')">${t}</button>`).join(\'\')}\n      </div>\n    </div>\n  `).join(\'\');\n}\nbuildHints();\n\nfunction appendTag(tag) {\n  const ta = document.getElementById(\'prompt-input\');\n  ta.value = ta.value.trim() ? ta.value.trim() + \', \' + tag : tag;\n}\n\nfunction toggleHints() {\n  const toggle = document.getElementById(\'hints-toggle\');\n  const panel  = document.getElementById(\'hints-panel\');\n  toggle.classList.toggle(\'open\');\n  panel.classList.toggle(\'open\');\n}\n\n// ── Examples ────────────────────────────────────────────────────────────────\nconst EXAMPLES = [\n  \'a hyperrealistic portrait photo of a young woman with brown hair, soft studio lighting, detailed skin texture, 8k resolution, sharp focus, neutral background, subtle makeup\',\n  \'a cinematic close-up of a middle-aged man with a short beard, dramatic side lighting, high contrast, film grain, ultra detailed face, shallow depth of field, 4k resolution\',\n  \'a portrait of an elderly woman with deep wrinkles and kind eyes, warm indoor lighting, high detail, natural skin tones, sharp focus, soft blurred background, 8k\',\n  \'a studio portrait of a young man with curly dark hair, confident expression, soft key light and rim light, highly detailed eyes, photorealistic, 4k ultra sharp\',\n];\n\nfunction buildExamples() {\n  document.getElementById(\'examples-list\').innerHTML = EXAMPLES.map((ex, i) => `\n    <button class="example-btn" onclick="useExample(${i})">\n      <span class="example-icon">◱</span>\n      <span class="example-text">${ex}</span>\n    </button>\n  `).join(\'\');\n}\nbuildExamples();\n\nlet appliedIdx = null;\nfunction useExample(i) {\n  document.getElementById(\'prompt-input\').value = EXAMPLES[i];\n  if (appliedIdx !== null) document.querySelectorAll(\'.example-btn\')[appliedIdx].querySelector(\'.example-text\').textContent = EXAMPLES[appliedIdx];\n  document.querySelectorAll(\'.example-btn\')[i].querySelector(\'.example-text\').textContent = \'✓ Applied\';\n  appliedIdx = i;\n  setTimeout(() => {\n    if (appliedIdx === i) document.querySelectorAll(\'.example-btn\')[i].querySelector(\'.example-text\').textContent = EXAMPLES[i];\n  }, 1500);\n}\n\n// ── Builder ──────────────────────────────────────────────────────────────────\nconst ATTRS = [\n  { key: \'shotType\',  label: \'Shot type\',  def: \'portrait photo\',\n    opts: [[\'portrait photo\',\'Portrait photo\'],[\'cinematic close-up\',\'Cinematic close-up\'],[\'studio portrait\',\'Studio portrait\'],[\'close-up portrait\',\'Close-up portrait\'],[\'editorial portrait\',\'Editorial portrait\']] },\n  { key: \'subject\',   label: \'Subject\',    def: \'a young woman\',\n    opts: [[\'a young woman\',\'Young woman\'],[\'a young man\',\'Young man\'],[\'a middle-aged woman\',\'Middle-aged woman\'],[\'a middle-aged man\',\'Middle-aged man\'],[\'an elderly woman\',\'Elderly woman\'],[\'an elderly man\',\'Elderly man\']] },\n  { key: \'hairColor\', label: \'Hair color\', def: \'brown hair\',\n    opts: [[\'brown hair\',\'Brown\'],[\'blonde hair\',\'Blonde\'],[\'black hair\',\'Black\'],[\'red hair\',\'Red\'],[\'gray hair\',\'Gray\'],[\'silver hair\',\'Silver\'],[\'auburn hair\',\'Auburn\']] },\n  { key: \'hairStyle\', label: \'Hair style\', def: \'straight hair\',\n    opts: [[\'straight hair\',\'Straight\'],[\'curly hair\',\'Curly\'],[\'wavy hair\',\'Wavy\'],[\'short hair\',\'Short\'],[\'long hair\',\'Long\'],[\'braided hair\',\'Braided\']] },\n  { key: \'expression\',label: \'Expression\', def: \'warm smile\',\n    opts: [[\'warm smile\',\'Warm smile\'],[\'smiling\',\'Smiling\'],[\'serious expression\',\'Serious\'],[\'neutral expression\',\'Neutral\'],[\'laughing\',\'Laughing\'],[\'kind eyes\',\'Kind eyes\']] },\n  { key: \'extras\',    label: \'Features\',   def: \'\',\n    opts: [[\'\',\'None\'],[\'wearing glasses\',\'Glasses\'],[\'with freckles\',\'Freckles\'],[\'with a short beard\',\'Short beard\'],[\'subtle makeup\',\'Subtle makeup\'],[\'wearing earrings\',\'Earrings\']] },\n  { key: \'lighting\',  label: \'Lighting\',   def: \'soft studio lighting\',\n    opts: [[\'soft studio lighting\',\'Soft studio\'],[\'dramatic side lighting\',\'Dramatic side\'],[\'natural daylight\',\'Natural daylight\'],[\'golden hour lighting\',\'Golden hour\'],[\'warm indoor lighting\',\'Warm indoor\'],[\'soft window light\',\'Window light\']] },\n  { key: \'style\',     label: \'Style\',      def: \'hyperrealistic\',\n    opts: [[\'hyperrealistic\',\'Hyperrealistic\'],[\'cinematic\',\'Cinematic\'],[\'film grain\',\'Film grain\'],[\'professional photography\',\'Professional\']] },\n  { key: \'quality\',   label: \'Quality\',    def: \'8k resolution, sharp focus, highly detailed\',\n    opts: [[\'8k resolution, sharp focus, highly detailed\',\'8K Ultra\'],[\'4k resolution, sharp focus\',\'4K High\'],[\'ultra detailed face, shallow depth of field\',\'Ultra detail\'],[\'detailed skin texture, sharp focus\',\'Detailed skin\']] },\n];\n\nfunction buildBuilderGrid() {\n  document.getElementById(\'builder-grid\').innerHTML = ATTRS.map(attr => `\n    <div class="select-group">\n      <div class="section-label">${attr.label}</div>\n      <div class="select-wrapper">\n        <select id="sel-${attr.key}" onchange="updateBuilderPrompt()">\n          ${attr.opts.map(([val,lbl]) => `<option value="${val}"${val===attr.def?\' selected\':\'\'}>${lbl}</option>`).join(\'\')}\n        </select>\n        <span class="select-arrow">▼</span>\n      </div>\n    </div>\n  `).join(\'\');\n  updateBuilderPrompt();\n}\nbuildBuilderGrid();\n\nfunction getBuilderValues() {\n  const v = {};\n  ATTRS.forEach(a => { v[a.key] = document.getElementById(\'sel-\' + a.key)?.value || \'\'; });\n  return v;\n}\n\nfunction buildPromptFromValues(v) {\n  return [\n    `a ${v.style} ${v.shotType}`,\n    `of ${v.subject}`,\n    `with ${v.hairColor}, ${v.hairStyle}`,\n    v.expression,\n    v.extras,\n    v.lighting,\n    v.quality,\n  ].filter(Boolean).join(\', \');\n}\n\nfunction updateBuilderPrompt() {\n  const prompt = buildPromptFromValues(getBuilderValues());\n  document.getElementById(\'builder-preview\').textContent = prompt;\n}\n\nfunction getActivePrompt() {\n  if (currentMode === \'manual\') return document.getElementById(\'prompt-input\').value.trim();\n  return buildPromptFromValues(getBuilderValues());\n}\n\n// ── Generation ───────────────────────────────────────────────────────────────\nlet currentImageUrl = null;\n\nasync function handleGenerate() {\n  const prompt = getActivePrompt();\n  if (!prompt) { alert(\'Please enter a prompt.\'); return; }\n\n  const btn     = document.getElementById(\'generate-btn\');\n  const label   = document.getElementById(\'btn-label\');\n  const empty   = document.getElementById(\'canvas-empty\');\n  const overlay = document.getElementById(\'canvas-generating\');\n  const img     = document.getElementById(\'generated-img\');\n  const errBox  = document.getElementById(\'error-box\');\n  const metaBar = document.getElementById(\'image-meta\');\n\n  btn.disabled = true;\n  label.textContent = \'Generating…\';\n  btn.querySelector(\'span\').innerHTML = \'<div class="spinner-sm"></div>\';\n\n  empty.style.display   = \'none\';\n  errBox.classList.remove(\'visible\');\n  overlay.classList.add(\'visible\');\n  img.style.display     = \'none\';\n  metaBar.classList.remove(\'visible\');\n\n  const startTime = Date.now();\n\n  try {\n    const res = await fetch(\'/generate\', {\n      method: \'POST\',\n      headers: { \'Content-Type\': \'application/x-www-form-urlencoded\' },\n      body: \'prompt=\' + encodeURIComponent(prompt),\n    });\n\n    if (!res.ok) throw new Error(await res.text() || \'Server error\');\n\n    const blob = await res.blob();\n    if (currentImageUrl) URL.revokeObjectURL(currentImageUrl);\n    currentImageUrl = URL.createObjectURL(blob);\n\n    const seed     = res.headers.get(\'X-Seed\') || \'—\';\n    const duration = ((Date.now() - startTime) / 1000).toFixed(1);\n\n    img.src           = currentImageUrl;\n    img.style.display = \'block\';\n    overlay.classList.remove(\'visible\');\n\n    document.getElementById(\'meta-seed\').textContent = seed;\n    document.getElementById(\'meta-time\').textContent = duration;\n    document.getElementById(\'download-btn\').href     = currentImageUrl;\n    document.getElementById(\'download-btn\').download = `portrait_${seed}.png`;\n    metaBar.classList.add(\'visible\');\n\n  } catch (e) {\n    overlay.classList.remove(\'visible\');\n    document.getElementById(\'error-msg\').textContent = e.message;\n    errBox.classList.add(\'visible\');\n  } finally {\n    btn.disabled = false;\n    label.textContent = \'Generate Portrait\';\n    btn.querySelector(\'span:first-child\').textContent = \'✦\';\n  }\n}\n</script>\n</body>\n</html>'

app = Flask(__name__)

@app.route('/')
def home():
    return render_template_string(HTML)

@app.route('/generate', methods=['POST'])
def generate():
    prompt = request.form.get('prompt', '').strip()
    if not prompt:
        return 'No prompt provided', 400
    try:
        img, seed = generate_image(prompt)
        buf = BytesIO()
        img.save(buf, format='PNG')
        buf.seek(0)
        from flask import Response
        response = Response(buf.getvalue(), mimetype='image/png')
        response.headers['X-Seed'] = str(seed)
        response.headers['Access-Control-Expose-Headers'] = 'X-Seed'
        return response
    except Exception as e:
        return str(e), 500

def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

Thread(target=run_flask, daemon=True).start()
time.sleep(3)

# ── Enter your ngrok token below ──────────────────────────────────────────
NGROK_TOKEN = '36G9Oh0AOyewS5VDweYorzMgwdo_2uDBmRp2t2tFgEet49fK5'  # <-- paste your token here: https://ngrok.com/
# ─────────────────────────────────────────────────────────────────────────

assert NGROK_TOKEN, 'Please paste your ngrok token in the NGROK_TOKEN variable above.'

conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()
tunnel = ngrok.connect(5000, bind_tls=True)

print('=' * 60)
print('  PORTRAIT STUDIO IS LIVE')
print('=' * 60)
print(f'\n  URL: {tunnel.public_url}')
print('\n  Share this link — anyone can open it in their browser.')
print('  Press the stop button to shut down.\n')
print('=' * 60)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


  PORTRAIT STUDIO IS LIVE

  URL: https://noneloquent-endocrinologic-doloris.ngrok-free.dev

  Share this link — anyone can open it in their browser.
  Press the stop button to shut down.

